In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

In [2]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [3]:
data = np.load("../models/traj_data.npz")
X_loaded = data["X"]
y_loaded = data["y"]
hours_loaded = data["hours"]

In [4]:
print(X_loaded.shape, y_loaded.shape, hours_loaded.shape)

(100000, 107, 64) (100000,) (100000, 21)


In [5]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_loaded)  # Converts goal_node IDs -> integers
num_classes = len(le.classes_)

print("Number of unique goal nodes:", num_classes)

Number of unique goal nodes: 230


In [6]:
num_nodes = len(np.unique(y_encoded))  # number of unique nodes

model = Sequential()
model.add(Masking(mask_value=0., input_shape=(X_loaded.shape[1], X_loaded.shape[2])))
model.add(LSTM(128, return_sequences=False))  # return_sequences=True if predicting sequences
model.add(Dense(num_nodes, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',  # integer labels
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

model.summary()

c:\miniforge3\envs\tf310\lib\site-packages\keras\src\layers\core\masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 107, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 230)            │        29,670 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,486 (501.90 KB)

 Trainable params: 128,486 (501.90 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_loaded,
    y_encoded,
    batch_size=128,
    epochs=1,
    validation_split=0.1
)

704/704 ━━━━━━━━━━━━━━━━━━━━ 251s 349ms/step - accuracy: 0.7698 - loss: 1.0266 - val_accuracy: 0.9210 - val_loss: 0.3104
